In [1]:
# Базовые библиотеки для воспроизводимости, анализа и удобного вывода результатов.
import os
import re
import sys
import random
import subprocess
from dataclasses import dataclass
from typing import Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import display, Markdown
from sklearn.feature_extraction.text import TfidfVectorizer

os.environ["TOKENIZERS_PARALLELISM"] = "false"


def safe_ensure_package(package_name: str, import_name: Optional[str] = None) -> bool:
    """Пытается импортировать пакет и при необходимости установить его через pip.
    Если установка не удалась, возвращает False, но не роняет ноутбук.
    """
    target = import_name or package_name
    try:
        __import__(target)
        return True
    except Exception:
        print(f"Пробуем установить пакет: {package_name}")
        try:
            subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package_name])
            __import__(target)
            return True
        except Exception as e:
            print(f"Не удалось подготовить пакет {package_name}: {e!r}")
            return False


FAISS_READY = safe_ensure_package("faiss-cpu", "faiss")

try:
    import faiss  # type: ignore
except Exception:
    faiss = None
    FAISS_READY = False


# sentence-transformers опционален: ноутбук умеет работать и без него.
SENTENCE_TRANSFORMERS_READY = safe_ensure_package("sentence-transformers", "sentence_transformers")

print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)
print("FAISS доступен:", FAISS_READY)
print("sentence-transformers доступен:", SENTENCE_TRANSFORMERS_READY)

Пробуем установить пакет: faiss-cpu
Пробуем установить пакет: sentence-transformers


c:\Users\Пётр\Desktop\IAIapp\IAI\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


NumPy: 2.4.1
Pandas: 3.0.0
FAISS доступен: True
sentence-transformers доступен: True


In [2]:
# Фиксируем seed и определяем устройство.
def set_seed(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)


set_seed(42)

try:
    import torch
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
except Exception:
    DEVICE = "cpu"

print("Устройство для работы:", DEVICE)

Устройство для работы: cpu


In [3]:
# Учебная база знаний.
documents: List[Dict[str, str]] = [
    {
        "doc_id": "doc_01",
        "title": "Эмбеддинги текстов",
        "text": (
            "Эмбеддинг – это векторное представление текста. "
            "Семантически похожие фразы оказываются близко в векторном пространстве. "
            "Это позволяет искать документы не только по совпадению слов, но и по смыслу запроса."
        ),
    },
    {
        "doc_id": "doc_02",
        "title": "FAISS и поиск ближайших соседей",
        "text": (
            "FAISS – библиотека для быстрого поиска ближайших соседей по векторам. "
            "Она полезна, когда в базе знаний много чанков и нужно быстро находить top-k наиболее похожих фрагментов."
        ),
    },
    {
        "doc_id": "doc_03",
        "title": "Чанкинг и overlap",
        "text": (
            "Чанкинг разбивает длинный документ на более короткие фрагменты. "
            "Если чанк слишком большой, retrieval может возвращать слишком общий контекст. "
            "Если чанк слишком маленький, смысл распадается. "
            "Overlap помогает не потерять информацию на границах соседних фрагментов."
        ),
    },
    {
        "doc_id": "doc_04",
        "title": "Оценка качества retrieval",
        "text": (
            "Качество retrieval нельзя оценивать только визуально. "
            "Обычно используют hit@k, recall@k и MRR. "
            "Эти метрики помогают понять, нашел ли retriever релевантный документ и насколько высоко он оказался в выдаче."
        ),
    },
    {
        "doc_id": "doc_05",
        "title": "Обновление базы знаний",
        "text": (
            "После добавления новых документов база знаний должна быть переиндексирована. "
            "Иначе retriever не увидит новые фрагменты. "
            "В production-системах обновление индекса может быть периодическим или событийным."
        ),
    },
    {
        "doc_id": "doc_06",
        "title": "Галлюцинации в RAG",
        "text": (
            "RAG снижает риск галлюцинаций, но не устраняет его полностью. "
            "Если retrieval вернул нерелевантные фрагменты или генератор исказил найденный факт, итоговый ответ все равно будет ошибочным."
        ),
    },
    {
        "doc_id": "doc_07",
        "title": "Промпт с контекстом",
        "text": (
            "В RAG-сценарии важно явно передавать модели найденный контекст и ограничивать ответ этим контекстом. "
            "Полезно просить систему отвечать только на основании источников и возвращать указание на использованные фрагменты."
        ),
    },
    {
        "doc_id": "doc_08",
        "title": "Метаданные и фильтрация",
        "text": (
            "Помимо текста, retrieval может учитывать метаданные: тип документа, дату, автора, подразделение или теги. "
            "Фильтрация по метаданным уменьшает область поиска и повышает точность извлечения."
        ),
    },
    {
        "doc_id": "doc_09",
        "title": "Реранжирование результатов",
        "text": (
            "После первичного retrieval можно применять реранжирование. "
            "Сначала быстрый retriever извлекает кандидатов, а затем более точная модель пересортировывает их по полезности для ответа."
        ),
    },
    {
        "doc_id": "doc_10",
        "title": "Гибридный поиск",
        "text": (
            "Гибридный поиск сочетает dense retrieval и классический лексический поиск. "
            "Он полезен, когда часть запросов требует смыслового сходства, а часть – точного совпадения терминов или аббревиатур."
        ),
    },
]

docs_df = pd.DataFrame(documents)
display(docs_df[["doc_id", "title"]])

,doc_id,title
0,doc_01,Эмбеддинги текстов
1,doc_02,FAISS и поиск ближайших соседей
2,doc_03,Чанкинг и overlap
3,doc_04,Оценка качества retrieval
4,doc_05,Обновление базы знаний
5,doc_06,Галлюцинации в RAG
6,doc_07,Промпт с контекстом
7,doc_08,Метаданные и фильтрация
8,doc_09,Реранжирование результатов
9,doc_10,Гибридный поиск


In [4]:
def chunk_text(text: str, chunk_size: int = 28, overlap: int = 8) -> List[str]:
    words = text.split()
    if chunk_size <= 0:
        raise ValueError("chunk_size должен быть положительным.")
    if overlap >= chunk_size:
        raise ValueError("overlap должен быть меньше chunk_size.")

    chunks = []
    step = chunk_size - overlap
    for start in range(0, len(words), step):
        end = start + chunk_size
        chunk_words = words[start:end]
        if not chunk_words:
            continue
        chunks.append(" ".join(chunk_words))
        if end >= len(words):
            break
    return chunks


class EmbeddingBackend:
    def fit_documents(self, texts: List[str]) -> np.ndarray:
        raise NotImplementedError

    def encode_queries(self, texts: List[str]) -> np.ndarray:
        raise NotImplementedError


class TfidfBackend(EmbeddingBackend):
    def __init__(self) -> None:
        self.vectorizer = TfidfVectorizer(ngram_range=(1, 2))
        self.backend_name = "TF-IDF (fallback)"

    def fit_documents(self, texts: List[str]) -> np.ndarray:
        matrix = self.vectorizer.fit_transform(texts)
        vectors = matrix.astype(np.float32).toarray()
        norms = np.linalg.norm(vectors, axis=1, keepdims=True) + 1e-12
        return vectors / norms

    def encode_queries(self, texts: List[str]) -> np.ndarray:
        matrix = self.vectorizer.transform(texts)
        vectors = matrix.astype(np.float32).toarray()
        norms = np.linalg.norm(vectors, axis=1, keepdims=True) + 1e-12
        return vectors / norms


class SentenceTransformersBackend(EmbeddingBackend):
    def __init__(self, model_name: str, device: str = "cpu") -> None:
        from sentence_transformers import SentenceTransformer  # type: ignore
        self.model_name = model_name
        self.model = SentenceTransformer(model_name, device=device)
        self.backend_name = f"SentenceTransformer: {model_name}"

    def fit_documents(self, texts: List[str]) -> np.ndarray:
        vectors = self.model.encode(
            texts,
            batch_size=16,
            show_progress_bar=False,
            normalize_embeddings=True,
            convert_to_numpy=True,
        )
        return vectors.astype(np.float32)

    def encode_queries(self, texts: List[str]) -> np.ndarray:
        vectors = self.model.encode(
            texts,
            batch_size=16,
            show_progress_bar=False,
            normalize_embeddings=True,
            convert_to_numpy=True,
        )
        return vectors.astype(np.float32)


def choose_backend(device: str = "cpu") -> EmbeddingBackend:
    # Опциональная попытка dense backend.
    if SENTENCE_TRANSFORMERS_READY:
        try:
            return SentenceTransformersBackend(
                model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2",
                device=device,
            )
        except Exception as e:
            print("Dense backend недоступен, переходим к TF-IDF.")
            print("Причина:", repr(e))
    return TfidfBackend()


@dataclass
class RetrieverArtifacts:
    backend_name: str
    chunks_df: pd.DataFrame
    chunk_vectors: np.ndarray
    backend: EmbeddingBackend
    index: object


def build_retriever(
    documents: List[Dict[str, str]],
    chunk_size: int = 28,
    overlap: int = 8,
    device: str = "cpu",
) -> RetrieverArtifacts:
    rows = []
    for doc in documents:
        chunks = chunk_text(doc["text"], chunk_size=chunk_size, overlap=overlap)
        for chunk_id, chunk_text_value in enumerate(chunks, start=1):
            rows.append(
                {
                    "doc_id": doc["doc_id"],
                    "title": doc["title"],
                    "chunk_id": f'{doc["doc_id"]}_chunk_{chunk_id:02d}',
                    "chunk_text": chunk_text_value,
                }
            )

    chunks_df = pd.DataFrame(rows)
    backend = choose_backend(device=device)
    chunk_vectors = backend.fit_documents(chunks_df["chunk_text"].tolist()).astype(np.float32)

    if FAISS_READY:
        index = faiss.IndexFlatIP(chunk_vectors.shape[1])  # type: ignore
        index.add(chunk_vectors)
    else:
        index = chunk_vectors

    return RetrieverArtifacts(
        backend_name=backend.backend_name,
        chunks_df=chunks_df,
        chunk_vectors=chunk_vectors,
        backend=backend,
        index=index,
    )


def search_chunks(query: str, artifacts: RetrieverArtifacts, top_k: int = 3) -> pd.DataFrame:
    query_vector = artifacts.backend.encode_queries([query]).astype(np.float32)

    if FAISS_READY:
        scores, indices = artifacts.index.search(query_vector, top_k)  # type: ignore
        scores = scores[0]
        indices = indices[0]
    else:
        similarities = (artifacts.chunk_vectors @ query_vector.T).reshape(-1)
        indices = np.argsort(-similarities)[:top_k]
        scores = similarities[indices]

    result = artifacts.chunks_df.iloc[indices].copy().reset_index(drop=True)
    result.insert(0, "rank", np.arange(1, len(result) + 1))
    result["score"] = scores
    return result[["rank", "score", "doc_id", "title", "chunk_id", "chunk_text"]]

In [5]:
artifacts = build_retriever(
    documents,
    chunk_size=24,
    overlap=6,
    device=DEVICE,
)

print("Используемый backend:", artifacts.backend_name)
print("Количество чанков:", len(artifacts.chunks_df))
display(artifacts.chunks_df.head())

c:\Users\Пётр\Desktop\IAIapp\IAI\.venv\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Пётр\.cache\huggingface\hub\models--sentence-transformers--paraphrase-multilingual-MiniLM-L12-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 7057.06it/s]
Be

Используемый backend: SentenceTransformer: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Количество чанков: 17


,doc_id,title,chunk_id,chunk_text
0,doc_01,Эмбеддинги текстов,doc_01_chunk_01,Эмбеддинг – это векторное представление текста...
1,doc_01,Эмбеддинги текстов,doc_01_chunk_02,"не только по совпадению слов, но и по смыслу з..."
2,doc_02,FAISS и поиск ближайших соседей,doc_02_chunk_01,FAISS – библиотека для быстрого поиска ближайш...
3,doc_02,FAISS и поиск ближайших соседей,doc_02_chunk_02,и нужно быстро находить top-k наиболее похожих...
4,doc_03,Чанкинг и overlap,doc_03_chunk_01,Чанкинг разбивает длинный документ на более ко...


In [6]:
sample_queries = [
    "Почему нужен чанкинг в RAG?",
    "Как оценивают качество retrieval?",
    "Зачем переиндексировать базу знаний?",
]

for query in sample_queries:
    display(Markdown(f"### Запрос: {query}"))
    display(search_chunks(query, artifacts=artifacts, top_k=3))

### Запрос: Почему нужен чанкинг в RAG?

,rank,score,doc_id,title,chunk_id,chunk_text
0,1,0.423389,doc_07,Промпт с контекстом,doc_07_chunk_01,В RAG-сценарии важно явно передавать модели на...
1,2,0.384225,doc_06,Галлюцинации в RAG,doc_06_chunk_01,"RAG снижает риск галлюцинаций, но не устраняет..."
2,3,0.322996,doc_03,Чанкинг и overlap,doc_03_chunk_01,Чанкинг разбивает длинный документ на более ко...


### Запрос: Как оценивают качество retrieval?

,rank,score,doc_id,title,chunk_id,chunk_text
0,1,0.717545,doc_04,Оценка качества retrieval,doc_04_chunk_01,Качество retrieval нельзя оценивать только виз...
1,2,0.695651,doc_08,Метаданные и фильтрация,doc_08_chunk_01,"Помимо текста, retrieval может учитывать метад..."
2,3,0.665814,doc_04,Оценка качества retrieval,doc_04_chunk_02,retriever релевантный документ и насколько выс...


### Запрос: Зачем переиндексировать базу знаний?

,rank,score,doc_id,title,chunk_id,chunk_text
0,1,0.651822,doc_05,Обновление базы знаний,doc_05_chunk_01,После добавления новых документов база знаний ...
1,2,0.455475,doc_10,Гибридный поиск,doc_10_chunk_02,часть – точного совпадения терминов или аббрев...
2,3,0.453705,doc_08,Метаданные и фильтрация,doc_08_chunk_01,"Помимо текста, retrieval может учитывать метад..."


In [7]:
def build_context_from_retrieval(query: str, artifacts: RetrieverArtifacts, top_k: int = 3) -> Tuple[str, pd.DataFrame]:
    retrieved = search_chunks(query, artifacts=artifacts, top_k=top_k)
    context_blocks = []

    for _, row in retrieved.iterrows():
        block = (
            f"[Источник: {row['doc_id']} | {row['title']} | score={row['score']:.4f}]\n"
            f"{row['chunk_text']}"
        )
        context_blocks.append(block)

    context = "\n\n".join(context_blocks)
    return context, retrieved

In [8]:
query = "Зачем после обновления базы знаний нужна переиндексация?"
context, retrieved_df = build_context_from_retrieval(query, artifacts=artifacts, top_k=3)

display(Markdown(f"### Запрос: {query}"))
display(retrieved_df)
print(context)

### Запрос: Зачем после обновления базы знаний нужна переиндексация?

,rank,score,doc_id,title,chunk_id,chunk_text
0,1,0.742087,doc_05,Обновление базы знаний,doc_05_chunk_01,После добавления новых документов база знаний ...
1,2,0.427378,doc_07,Промпт с контекстом,doc_07_chunk_02,на основании источников и возвращать указание ...
2,3,0.421123,doc_03,Чанкинг и overlap,doc_03_chunk_01,Чанкинг разбивает длинный документ на более ко...


[Источник: doc_05 | Обновление базы знаний | score=0.7421]
После добавления новых документов база знаний должна быть переиндексирована. Иначе retriever не увидит новые фрагменты. В production-системах обновление индекса может быть периодическим или событийным.

[Источник: doc_07 | Промпт с контекстом | score=0.4274]
на основании источников и возвращать указание на использованные фрагменты.

[Источник: doc_03 | Чанкинг и overlap | score=0.4211]
Чанкинг разбивает длинный документ на более короткие фрагменты. Если чанк слишком большой, retrieval может возвращать слишком общий контекст. Если чанк слишком маленький, смысл распадается.


In [9]:
def generate_answer_from_context(query: str, context: str, max_sentences: int = 2) -> str:
    # Убираем технические строки источников из ранжирования, но не из общего контекста.
    raw_lines = [line.strip() for line in context.splitlines() if line.strip()]
    content_lines = [line for line in raw_lines if not line.startswith("[Источник:")]

    sentence_pool = []
    for line in content_lines:
        sentence_pool.extend(split_into_sentences(line))

    sentence_pool = [s for s in sentence_pool if len(s.split()) >= 4]

    if not sentence_pool:
        return "Недостаточно контекста для построения ответа."

    vectorizer = TfidfVectorizer(ngram_range=(1, 2))
    matrix = vectorizer.fit_transform([query] + sentence_pool).toarray().astype(np.float32)

    query_vec = matrix[0]
    sentence_vecs = matrix[1:]

    query_norm = np.linalg.norm(query_vec) + 1e-12
    sent_norms = np.linalg.norm(sentence_vecs, axis=1) + 1e-12
    scores = (sentence_vecs @ query_vec) / (sent_norms * query_norm)

    ranked_idx = np.argsort(-scores)
    selected_sentences = []
    used_normalized = set()

    for idx in ranked_idx:
        sentence = sentence_pool[idx]
        normalized = sentence.lower().strip()
        if scores[idx] <= 0:
            continue
        if normalized in used_normalized:
            continue
        used_normalized.add(normalized)
        selected_sentences.append(sentence)
        if len(selected_sentences) >= max_sentences:
            break

    if not selected_sentences:
        return "В найденном контексте нет достаточно релевантного фрагмента для уверенного ответа."

    return " ".join(selected_sentences)

In [11]:
def split_into_sentences(text: str) -> List[str]:
    parts = re.split(r"(?<=[.!?])\s+", text.strip())
    return [p.strip() for p in parts if p.strip()]


def pick_best_sentences(query: str, text: str, top_n: int = 2) -> List[str]:
    sentences = split_into_sentences(text)
    if not sentences:
        return []

    vectorizer = TfidfVectorizer(ngram_range=(1, 2))
    matrix = vectorizer.fit_transform([query] + sentences).toarray().astype(np.float32)

    query_vec = matrix[0]
    sentence_vecs = matrix[1:]

    query_norm = np.linalg.norm(query_vec) + 1e-12
    sent_norms = np.linalg.norm(sentence_vecs, axis=1) + 1e-12
    scores = (sentence_vecs @ query_vec) / (sent_norms * query_norm)

    best_idx = np.argsort(-scores)[:top_n]
    return [sentences[i] for i in best_idx if scores[i] > 0]


def answer_without_retrieval(query: str, documents: List[Dict[str, str]]) -> Dict[str, object]:
    doc_texts = [doc["title"] + ". " + doc["text"] for doc in documents]
    vectorizer = TfidfVectorizer(ngram_range=(1, 2))
    matrix = vectorizer.fit_transform(doc_texts + [query]).toarray().astype(np.float32)

    doc_vecs = matrix[:-1]
    query_vec = matrix[-1]

    doc_norms = np.linalg.norm(doc_vecs, axis=1) + 1e-12
    query_norm = np.linalg.norm(query_vec) + 1e-12
    scores = (doc_vecs @ query_vec) / (doc_norms * query_norm)

    best_idx = int(np.argmax(scores))
    best_doc = documents[best_idx]
    best_sentences = pick_best_sentences(query, best_doc["text"], top_n=2)

    if best_sentences:
        answer = " ".join(best_sentences)
    else:
        answer = (
            "Не удалось уверенно извлечь ответ без retrieval по чанкам. "
            "Система выбрала наиболее похожий документ целиком."
        )

    return {
        "answer": answer,
        "selected_doc_id": best_doc["doc_id"],
        "selected_title": best_doc["title"],
        "score": float(scores[best_idx]),
    }

In [12]:
answer_example = generate_answer_from_context(query, context)
print(answer_example)

После добавления новых документов база знаний должна быть переиндексирована.


In [13]:
def mini_rag_answer(
    query: str,
    artifacts: RetrieverArtifacts,
    top_k: int = 3,
    max_answer_sentences: int = 2,
) -> Dict[str, object]:
    context, retrieved = build_context_from_retrieval(query, artifacts=artifacts, top_k=top_k)
    answer = generate_answer_from_context(query, context=context, max_sentences=max_answer_sentences)

    return {
        "query": query,
        "answer": answer,
        "context": context,
        "sources": retrieved,
    }

In [14]:
comparison_queries = [
    "Почему overlap полезен при чанкинге?",
    "Как оценивают качество retrieval?",
    "Что делать после обновления базы знаний?",
    "Когда полезен гибридный поиск?",
    "Как RAG связан с галлюцинациями?",
]

comparison_rows = []

for query in comparison_queries:
    baseline = answer_without_retrieval(query, documents)
    rag = mini_rag_answer(query, artifacts=artifacts, top_k=3)

    comparison_rows.append(
        {
            "query": query,
            "baseline_doc_id": baseline["selected_doc_id"],
            "baseline_score": baseline["score"],
            "baseline_answer": baseline["answer"],
            "rag_answer": rag["answer"],
        }
    )

comparison_df = pd.DataFrame(comparison_rows)
display(comparison_df)

,query,baseline_doc_id,baseline_score,baseline_answer,rag_answer
0,Почему overlap полезен при чанкинге?,doc_03,0.055501,Overlap помогает не потерять информацию на гра...,Overlap помогает не потерять информацию на гра...
1,Как оценивают качество retrieval?,doc_04,0.114098,Качество retrieval нельзя оценивать только виз...,Качество retrieval нельзя оценивать только виз...
2,Что делать после обновления базы знаний?,doc_05,0.138992,После добавления новых документов база знаний ...,После добавления новых документов база знаний ...
3,Когда полезен гибридный поиск?,doc_10,0.359952,Гибридный поиск сочетает dense retrieval и кла...,Гибридный поиск сочетает dense retrieval и кла...
4,Как RAG связан с галлюцинациями?,doc_06,0.062478,"RAG снижает риск галлюцинаций, но не устраняет...","RAG снижает риск галлюцинаций, но не устраняет..."


In [15]:
rag_result = mini_rag_answer(
    "Почему overlap полезен при чанкинге?",
    artifacts=artifacts,
    top_k=3,
)

display(Markdown(f"### Вопрос: {rag_result['query']}"))
display(Markdown(f"**Ответ:** {rag_result['answer']}"))
display(Markdown("**Источники:**"))
display(rag_result["sources"])

### Вопрос: Почему overlap полезен при чанкинге?

**Ответ:** Overlap помогает не потерять информацию на границах соседних фрагментов.

**Источники:**

,rank,score,doc_id,title,chunk_id,chunk_text
0,1,0.755862,doc_03,Чанкинг и overlap,doc_03_chunk_02,"Если чанк слишком маленький, смысл распадается..."
1,2,0.530003,doc_10,Гибридный поиск,doc_10_chunk_02,часть – точного совпадения терминов или аббрев...
2,3,0.440923,doc_03,Чанкинг и overlap,doc_03_chunk_01,Чанкинг разбивает длинный документ на более ко...


In [16]:
qa_benchmark = [
    {
        "query_id": "q01",
        "query": "Почему overlap полезен при чанкинге?",
        "relevant_doc_ids": ["doc_03"],
        "expected_keywords": ["overlap", "границах", "соседних", "фрагментов"],
    },
    {
        "query_id": "q02",
        "query": "Как оценивают качество retrieval?",
        "relevant_doc_ids": ["doc_04"],
        "expected_keywords": ["hit@k", "recall@k", "MRR"],
    },
    {
        "query_id": "q03",
        "query": "Что делать после обновления базы знаний?",
        "relevant_doc_ids": ["doc_05"],
        "expected_keywords": ["переиндексирована", "новые", "документы"],
    },
    {
        "query_id": "q04",
        "query": "Когда полезен гибридный поиск?",
        "relevant_doc_ids": ["doc_10"],
        "expected_keywords": ["dense", "лексический", "терминов"],
    },
    {
        "query_id": "q05",
        "query": "Зачем нужен реранжировщик после первичного retrieval?",
        "relevant_doc_ids": ["doc_09"],
        "expected_keywords": ["кандидатов", "пересортировывает", "полезности"],
    },
]


def keyword_recall(answer: str, expected_keywords: List[str]) -> float:
    answer_lower = answer.lower()
    hits = sum(1 for kw in expected_keywords if kw.lower() in answer_lower)
    return hits / len(expected_keywords) if expected_keywords else np.nan


def evaluate_mini_rag(
    benchmark_rows: List[Dict[str, object]],
    artifacts: RetrieverArtifacts,
    top_k: int = 3,
) -> pd.DataFrame:
    rows = []

    for item in benchmark_rows:
        query = item["query"]
        relevant_doc_ids = item["relevant_doc_ids"]
        expected_keywords = item["expected_keywords"]

        retrieved = search_chunks(query, artifacts=artifacts, top_k=top_k)
        predicted_doc_ids = retrieved["doc_id"].tolist()
        retrieval_hit = int(any(doc_id in predicted_doc_ids for doc_id in relevant_doc_ids))

        baseline = answer_without_retrieval(query, documents)
        rag = mini_rag_answer(query, artifacts=artifacts, top_k=top_k)

        rows.append(
            {
                "query_id": item["query_id"],
                "query": query,
                "relevant_doc_ids": ", ".join(relevant_doc_ids),
                "predicted_doc_ids": ", ".join(predicted_doc_ids),
                f"retrieval_hit@{top_k}": retrieval_hit,
                "baseline_keyword_recall": keyword_recall(baseline["answer"], expected_keywords),
                "rag_keyword_recall": keyword_recall(rag["answer"], expected_keywords),
                "baseline_answer": baseline["answer"],
                "rag_answer": rag["answer"],
            }
        )

    return pd.DataFrame(rows)

In [17]:
evaluation_df = evaluate_mini_rag(qa_benchmark, artifacts=artifacts, top_k=3)
display(evaluation_df)

,query_id,query,relevant_doc_ids,predicted_doc_ids,retrieval_hit@3,baseline_keyword_recall,rag_keyword_recall,baseline_answer,rag_answer
0,q01,Почему overlap полезен при чанкинге?,doc_03,"doc_03, doc_10, doc_03",1,1.000000,1.000000,Overlap помогает не потерять информацию на гра...,Overlap помогает не потерять информацию на гра...
1,q02,Как оценивают качество retrieval?,doc_04,"doc_04, doc_08, doc_04",1,0.000000,0.000000,Качество retrieval нельзя оценивать только виз...,Качество retrieval нельзя оценивать только виз...
2,q03,Что делать после обновления базы знаний?,doc_05,"doc_05, doc_03, doc_07",1,0.333333,0.333333,После добавления новых документов база знаний ...,После добавления новых документов база знаний ...
3,q04,Когда полезен гибридный поиск?,doc_10,"doc_10, doc_09, doc_02",1,1.000000,1.000000,Гибридный поиск сочетает dense retrieval и кла...,Гибридный поиск сочетает dense retrieval и кла...
4,q05,Зачем нужен реранжировщик после первичного ret...,doc_09,"doc_09, doc_08, doc_04",1,0.000000,0.000000,После первичного retrieval можно применять рер...,После первичного retrieval можно применять рер...
